# 19 — Design Patterns

## Objectives
- Implement Creational, Structural, and Behavioral patterns
- Know when to apply each pattern
- Recognize patterns in Java standard library

## Pattern Categories
| Category | Patterns | Purpose |
|----------|----------|---------|  
| Creational | Singleton, Factory, Builder | Object creation |
| Structural | Adapter, Decorator, Facade | Object composition |
| Behavioral | Observer, Strategy, Command | Object communication |

In [1]:
// SINGLETON — thread-safe double-checked locking
class DatabaseConnection {
    private static volatile DatabaseConnection instance;
    private String url;
    
    private DatabaseConnection() {
        this.url = "jdbc:mysql://localhost:3306/mydb";
        System.out.println("DB connection created: " + url);
    }
    
    public static DatabaseConnection getInstance() {
        if (instance == null) {
            synchronized (DatabaseConnection.class) {
                if (instance == null) instance = new DatabaseConnection();
            }
        }
        return instance;
    }
    
    public void query(String sql) { System.out.println("Executing: " + sql); }
}

// FACTORY — object creation without specifying exact class
interface Animal { void speak(); }
class Dog implements Animal { public void speak() { System.out.println("Woof!"); } }
class Cat implements Animal { public void speak() { System.out.println("Meow!"); } }
class Cow implements Animal { public void speak() { System.out.println("Moo!"); } }

class AnimalFactory {
    public static Animal create(String type) {
        return switch (type.toLowerCase()) {
            case "dog" -> new Dog();
            case "cat" -> new Cat();
            case "cow" -> new Cow();
            default -> throw new IllegalArgumentException("Unknown: " + type);
        };
    }
}

// OBSERVER — event notification
interface EventListener { void onEvent(String event, Object data); }

class EventBus {
    private java.util.Map<String, java.util.List<EventListener>> listeners = new java.util.HashMap<>();
    
    void subscribe(String event, EventListener listener) {
        listeners.computeIfAbsent(event, k -> new java.util.ArrayList<>()).add(listener);
    }
    
    void publish(String event, Object data) {
        listeners.getOrDefault(event, java.util.Collections.emptyList())
                 .forEach(l -> l.onEvent(event, data));
    }
}

// Demo all patterns
System.out.println("=== Singleton ===");
DatabaseConnection.getInstance().query("SELECT * FROM users");
DatabaseConnection.getInstance().query("SELECT * FROM orders");
System.out.println("Same instance: " + (DatabaseConnection.getInstance() == DatabaseConnection.getInstance()));

System.out.println("\n=== Factory ===");
String[] types = {"dog", "cat", "cow"};
for (String type : types) AnimalFactory.create(type).speak();

System.out.println("\n=== Observer ===");
EventBus bus = new EventBus();
bus.subscribe("ORDER_PLACED", (e, d) -> System.out.println("[Email] Order notification: " + d));
bus.subscribe("ORDER_PLACED", (e, d) -> System.out.println("[SMS] Order alert: " + d));
bus.subscribe("USER_LOGIN",   (e, d) -> System.out.println("[Audit] Login: " + d));
bus.publish("ORDER_PLACED", "ORD-2024-001");
bus.publish("USER_LOGIN", "alice@example.com");

=== Singleton ===
DB connection created: jdbc:mysql://localhost:3306/mydb
Executing: SELECT * FROM users
Executing: SELECT * FROM orders
Same instance: true

=== Factory ===
Woof!
Meow!
Moo!

=== Observer ===
[Email] Order notification: ORD-2024-001
[SMS] Order alert: ORD-2024-001
[Audit] Login: alice@example.com


## Design Patterns in Java Standard Library
| Pattern | Java Example |
|---------|-------------|
| Singleton | `Runtime.getRuntime()` |
| Factory | `Calendar.getInstance()` |
| Iterator | `Iterator<E>` |
| Observer | `java.util.Observer` |
| Decorator | `BufferedReader(FileReader)` |
| Strategy | `Comparator<T>` |

## Mini Challenge
Implement a `NotificationBuilder` using the **Builder** pattern that can configure: email, SMS, push, priority level, and retry count.

In [5]:
// Everything combined inside a single class so it runs perfectly in a single Jupyter cell
public class BuilderDemo {

    // 1. The Product Class (Nested inside the main class)
    static class Notification {
        private final String email;
        private final String sms;
        private final String push;
        private final String priorityLevel;
        private final int retryCount;

        private Notification(NotificationBuilder builder) {
            this.email = builder.email;
            this.sms = builder.sms;
            this.push = builder.push;
            this.priorityLevel = builder.priorityLevel;
            this.retryCount = builder.retryCount;
        }

        @Override
        public String toString() {
            return "Notification {" +
                   "email='" + email + '\'' +
                   ", sms='" + sms + '\'' +
                   ", push='" + push + '\'' +
                   ", priorityLevel='" + priorityLevel + '\'' +
                   ", retryCount=" + retryCount +
                   '}';
        }

        // 2. The Static Builder Class
        public static class NotificationBuilder {
            private String email;
            private String sms;
            private String push;
            private String priorityLevel = "LOW"; 
            private int retryCount = 0;           

            public NotificationBuilder setEmail(String email) {
                this.email = email;
                return this; 
            }

            public NotificationBuilder setSms(String sms) {
                this.sms = sms;
                return this;
            }

            public NotificationBuilder setPush(String push) {
                this.push = push;
                return this;
            }

            public NotificationBuilder setPriorityLevel(String priorityLevel) {
                this.priorityLevel = priorityLevel;
                return this;
            }

            public NotificationBuilder setRetryCount(int retryCount) {
                this.retryCount = retryCount;
                return this;
            }

            public Notification build() {
                return new Notification(this);
            }
        }
    }

    // 3. Execution Main Method
    public static void main(String[] args) {
        System.out.println("=== Builder Pattern ===");

        // Configure a high-priority email + push notification with retries
        Notification criticalAlert = new Notification.NotificationBuilder()
                .setEmail("admin@example.com")
                .setPush("Device_ID_9921")
                .setPriorityLevel("HIGH")
                .setRetryCount(3)
                .build();

        // Configure a simple SMS notification using defaults for the rest
        Notification quickSms = new Notification.NotificationBuilder()
                .setSms("+15550199")
                .build();

        System.out.println(criticalAlert);
        System.out.println(quickSms);
    }
}

// Call the main method directly so Jupyter executes it instantly
BuilderDemo.main(new String[0]);

=== Builder Pattern ===
Notification {email='admin@example.com', sms='null', push='Device_ID_9921', priorityLevel='HIGH', retryCount=3}
Notification {email='null', sms='+15550199', push='null', priorityLevel='LOW', retryCount=0}
